In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from transformer_lens import HookedTransformer

import json
import time
import os

from src.neuron_texts import find_neurons

In [3]:
# Parameters
# Alternative model/layer presets used in the paper experiments:

model_name = "gpt2-small"; 
layers = [9, 10, 11]

# model_name = "gpt2-medium"
# layers = [20, 21, 22, 23]  # last 4 layers

#model_name = "gpt2-large"
#layers = [31, 32, 33, 34, 35]

device = "cpu"
# layers = [9, 10, 11]
# layers = [20, 21, 22, 23]
# layers = [31, 32, 33, 34, 35]
num_neurons_per_layer = 20
rejected_neurons = [()]

In [4]:
#!export PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True

In [5]:
model = HookedTransformer.from_pretrained(
    model_name,
    center_unembed=True,
    center_writing_weights=True,
    fold_ln=True,
    # refactor_factored_attn_matrices=True,
    device=device,
)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loaded pretrained model gpt2-small into HookedTransformer


In [6]:
import random
from fancy_einsum import einsum
import torch

def find_random_neurons(model, layers, num_neurons_per_layer, rejected_neurons):
    results = {}
    rejected_neurons_with_tokens = []

    # Handle layers if they are negative
    num_layers = model.cfg.n_layers
    layers = [layer if layer >= 0 else num_layers + layer for layer in layers]

    for layer_num in layers:
        n_layer_neurons = model.W_out[layer_num, :, :]
        unembedding = model.W_U
        dot_product = einsum("neuron embed, embed token -> neuron token", n_layer_neurons, unembedding)
        
        values, indices = torch.max(dot_product, dim=-1) # Get the highest congruence with any given token for each neuron
        top_values, top_indices = torch.topk(values, indices.shape[0])
        
        found_i = []
        while len(found_i) < num_neurons_per_layer:
            i = random.randint(0, len(top_indices))
            if i in found_i:
                continue
            else:
                str_token = model.to_string(indices[top_indices][i])
                found_i.append(i)
                neuron_index = top_indices[i].item()
                results[(layer_num, neuron_index)] = {'token': str_token, 'congruence': top_values[i].item()}

    return results, None

In [7]:
# rejected_neurons = [(9, 2322), (9, 2894), (11, 3033)] # GPT-2 Small, 20 neurons per layer
# rejected_neurons = [(23, 3671), (23, 3177)] # GPT-2 Medium, 20 neurons per layer
# rejected_neurons = [(31, 4172), (31, 4899), (32, 4361), (33, 1582), (33, 122), (33, 2587), 
#                    (34, 2978), (34, 805), (35, 274), (35, 920), (35, 4849), (35, 684), 
#                    (35, 4396), (35, 295), (35, 3065), (35, 989)]  # GPT-2 Large, 40 neurons per layer
# rejected_neurons = [(44, 1657), (45, 284),(45, 4415) ,(46, 1785),(46, 1795),(47, 2065),
                    # (47, 1147),(47, 3489), (47, 4845),(47, 5429),(47, 4019),
                    # (43, 403),(47, 6356),(47, 3775),(47, 3635),] # GPT-2 XL, 20 neurons per layer

# rejected_neurons = [(16, 2727), (16, 523), (17, 4787),(18, 2078),(18, 123),(18, 4681),(19, 5095),(19, 4034),(20, 2259),(16, 4244),(20,795)] # GPT-2-large mid

# rejected_neurons = [(9, 2871), (10, 741), (10, 336), (11, 1266) , (11, 493) ,(11, 817) ,(11, 922),(11, 2790),(11, 1953) ,(11, 2209) ,(11, 2631) ,(11, 2640) ,(11, 803) ,(11, 875) ,(11, 2856),(11, 1240) ,(11, 339),(11, 1248),(11, 1994),
                    # (11, 1828),(11, 1076),(11, 1347), (11,110), (11,45), (11, 334), (11,1932)] # pythia 160m
# rejected_neurons = [(10, 457), (11,2627)]


# Update this until you are satisfied with the results

neuron_results, rejected_neurons_with_token = find_neurons(
    model=model,
    layers=layers,
    num_neurons_per_layer=num_neurons_per_layer,
    rejected_neurons=rejected_neurons,
)

for k, v in neuron_results.items():
    print(k, f"'{v['token']}'")

(9, 5) ' too'
(9, 1616) ' each'
(9, 1985) ' because'
(9, 1496) ' such'
(9, 2513) ' term'
(9, 1047) ' does'
(9, 2209) ' number'
(9, 1690) ' by'
(9, 1272) ' March'
(9, 2250) ' while'
(9, 947) ' times'
(9, 1360) ' they'
(9, 2322) ' "''
(9, 190) ' them'
(9, 2125) ' example'
(9, 202) ' that'
(9, 1308) ' every'
(9, 1341) ' well'
(9, 2151) ' check'
(9, 2894) ' 1870'
(10, 3041) ' right'
(10, 1122) ' way'
(10, 2124) ' there'
(10, 1687) ' out'
(10, 2714) ' one'
(10, 1272) ' so'
(10, 2467) ' than'
(10, 211) ' up'
(10, 1453) ' then'
(10, 2680) ' today'
(10, 2408) ' over'
(10, 2003) ' so'
(10, 297) ' your'
(10, 2093) ' themselves'
(10, 1315) ' down'
(10, 1505) ' then'
(10, 3032) ' it'
(10, 2522) ' after'
(10, 304) ' state'
(10, 2149) ' like'
(11, 2003) ' work'
(11, 1605) ' about'
(11, 948) ' right'
(11, 2069) ' off'
(11, 2274) ' them'
(11, 2465) ' right'
(11, 747) ' her'
(11, 2040) ' or'
(11, 1406) ' only'
(11, 2957) ' on'
(11, 967) ' called'
(11, 3015) ' on'
(11, 822) ' time'
(11, 2586) ' like'
(1

In [8]:
random_neuron_results, _ = find_random_neurons(
    model=model,
    layers=layers,
    num_neurons_per_layer=num_neurons_per_layer,
    rejected_neurons=rejected_neurons,
)

for k, v in random_neuron_results.items():
    print(k, f"'{v['token']}'")

(9, 327) 'been'
(9, 711) 'soType'
(9, 1331) 'hours'
(9, 902) '══'
(9, 1543) '="#'
(9, 2010) ' Justice'
(9, 1861) ' subject'
(9, 2461) 'o'
(9, 2872) 'Clar'
(9, 1670) 'graduate'
(9, 226) 'rera'
(9, 465) '),"'
(9, 498) 'DVD'
(9, 812) 'SPONSORED'
(9, 1122) ' People'
(9, 2639) ' Reeves'
(9, 1065) ' agreement'
(9, 199) 'ディ'
(9, 533) ' can'
(9, 662) ' LSU'
(10, 1572) 'contact'
(10, 1145) '.—'
(10, 452) 'Offline'
(10, 2408) ' over'
(10, 1862) 'iliary'
(10, 1089) 'ascript'
(10, 2749) ' withd'
(10, 2358) ' Rounds'
(10, 1803) ' man'
(10, 393) ' say'
(10, 2080) 'owship'
(10, 1600) 'nic'
(10, 310) 'ufact'
(10, 2661) 'ال'
(10, 1421) ' Logged'
(10, 721) ' got'
(10, 2981) ' Lever'
(10, 457) ' Sloven'
(10, 1521) 'Done'
(10, 440) 'a'
(11, 2540) ' Monkey'
(11, 2165) 'kat'
(11, 2781) ' Resp'
(11, 1656) ' comprom'
(11, 1051) 'actionDate'
(11, 3046) 'Going'
(11, 441) 'iquid'
(11, 393) 'nels'
(11, 1924) ' F'
(11, 2101) ' Aad'
(11, 1269) 'ansion'
(11, 1327) 'ilial'
(11, 440) 'abal'
(11, 2641) '──'
(11, 2521) 

In [9]:
## Output of this notebook
output = {
    'parameters': {
        'model_name': model_name,
        'layers': layers,
        'num_neurons_per_layer': num_neurons_per_layer,
        'rejected_neurons': rejected_neurons_with_token,
    },
    'neurons': {str(k): v for k,v in neuron_results.items()},
}

# Save the output to a in ../data/neurons/
timestamp = time.strftime("%Y-%m-%d_%H-%M-%S", time.localtime(int(time.time())))
filename = f"{timestamp}_{model_name}.json"

folder_path = "../experiment_data/1_next_token_neurons"
filepath = os.path.join(folder_path, filename)
if os.path.isfile(filepath):
    raise Exception("File already exists!")
os.makedirs(folder_path, exist_ok=True)

with open(filepath, 'w') as f:
    json.dump(output, f, indent=4)

In [10]:
## Output of this notebook, for random neuron
output = {
    'parameters': {
        'model_name': model_name,
        'layers': layers,
        'num_neurons_per_layer': num_neurons_per_layer,
        # 'rejected_neurons': rejected_neurons_with_token,
        'rejected_neurons': [],
    },
    'neurons': {str(k): v for k,v in random_neuron_results.items()},
}

# Save the output to a in ../data/neurons/
timestamp = time.strftime("%Y-%m-%d_%H-%M-%S", time.localtime(int(time.time())))
filename = f"{timestamp}_{model_name}_random.json"

folder_path = "../experiment_data/1_next_token_neurons"
filepath = os.path.join(folder_path, filename)
if os.path.isfile(filepath):
    raise Exception("File already exists!")
os.makedirs(folder_path, exist_ok=True)

with open(filepath, 'w') as f:
    json.dump(output, f, indent=4)